In [1]:
import pandas as pd

# Recharger les données brutes
df = pd.read_csv("../data/raw/superstore_raw.csv", encoding="latin-1")

# --- Correction 1 : Postal Code (zéro initial perdu) ---
df["Postal Code"] = df["Postal Code"].astype(str).str.zfill(5)

# Vérification
assert df["Postal Code"].str.len().eq(5).all(), "Certains codes postaux ne font pas 5 caractères"
print("Postal Code : OK, tous les codes font 5 caractères")

Postal Code : OK, tous les codes font 5 caractères


In [3]:
# --- Correction 2 : conversion des dates ---
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%m/%d/%Y")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], format="%m/%d/%Y")

# Vérification rapide
print(df[["Order Date", "Ship Date"]].dtypes)
print(f"\nPériode couverte : {df['Order Date'].min()} -> {df['Order Date'].max()}")

Order Date    datetime64[ns]
Ship Date     datetime64[ns]
dtype: object

Période couverte : 2014-01-03 00:00:00 -> 2017-12-30 00:00:00


In [5]:
# --- Vérification de cohérence métier : Ship Date >= Order Date ---
incoherences_dates = (df["Ship Date"] < df["Order Date"]).sum()
print(f"Lignes où la livraison précède la commande : {incoherences_dates}")

Lignes où la livraison précède la commande : 0


In [7]:
# --- Vérification : valeurs négatives suspectes ---
print(f"Sales négatifs : {(df['Sales'] < 0).sum()}")
print(f"Quantity négatifs ou nuls : {(df['Quantity'] <= 0).sum()}")
print(f"Discount hors [0, 1] : {((df['Discount'] < 0) | (df['Discount'] > 1)).sum()}")

Sales négatifs : 0
Quantity négatifs ou nuls : 0
Discount hors [0, 1] : 0


In [11]:
print(f"Lignes avec Profit négatif : {(df['Profit'] < 0).sum()}")
print(f"Part du CA total concernée : {df[df['Profit']<0]['Sales'].sum() / df['Sales'].sum() * 100:.1f}%")

Lignes avec Profit négatif : 1871
Part du CA total concernée : 20.4%


In [13]:
print(f"Nombre de lignes : {len(df)}")
print(f"Nombre d'Order ID uniques : {df['Order ID'].nunique()}")
print(f"Une commande contient en moyenne {len(df) / df['Order ID'].nunique():.2f} lignes")

Nombre de lignes : 9994
Nombre d'Order ID uniques : 5009
Une commande contient en moyenne 2.00 lignes


In [15]:
variants = df.groupby("Product ID")["Product Name"].nunique()
nb_variants = (variants > 1).sum()
print(f"Product ID avec plusieurs noms différents : {nb_variants}")

# Regarder quelques exemples concrets
ids_a_variantes = variants[variants > 1].index
df[df["Product ID"].isin(ids_a_variantes)][["Product ID", "Product Name"]] \
    .drop_duplicates() \
    .sort_values("Product ID") \
    .head(20)

Product ID avec plusieurs noms différents : 32


,Product ID,Product Name
2471,FUR-BO-10002213,"Sauder Forest Hills Library, Woodland Oak Finish"
2115,FUR-BO-10002213,DMI Eclipse Executive Suite Bookcases
66,FUR-CH-10001146,"Global Value Mid-Back Manager's Chair, Gray"
128,FUR-CH-10001146,"Global Task Chair, Black"
1459,FUR-FU-10001473,DAX Wood Document Frame
2204,FUR-FU-10001473,"Eldon Executive Woodline II Desk Accessories, ..."
3831,FUR-FU-10004017,"Executive Impressions 13"" Chairman Wall Clock"
234,FUR-FU-10004017,Tenex Contemporary Contur Chairmats for Low an...
1385,FUR-FU-10004091,"Eldon 200 Class Desk Accessories, Black"
293,FUR-FU-10004091,"Howard Miller 13"" Diameter Goldtone Round Wall..."


In [9]:
# --- Export du fichier nettoyé ---
df.to_csv("../data/processed/superstore_clean.csv", index=False, encoding="utf-8")
print(f"Export terminé : {df.shape[0]} lignes, {df.shape[1]} colonnes")

Export terminé : 9994 lignes, 21 colonnes


In [17]:
print(round(df["Sales"].sum(), 2))

2297200.86
